# Training on INCLUDE_50 Dataset

## Preprocessing Data

In [1]:
import pandas as pd
import numpy as np
import os

from tqdm.notebook import tqdm

In [2]:
train_data = pd.read_csv('Dataset/train.csv')
test_data = pd.read_csv('Dataset/test.csv')

train_data.sample(5)

,parent_label,label,video_path,include_50
3675,Greetings,55. Thank you,Greetings/55. Thank you/MVI_0008.mp4,True
845,Home,28. Window,Home/28. Window/MVI_9015.MP4,False
255,Places,26. University,Places/26. University/MVI_3624.mp4,False
115,Means_of_Transportation,9. Train,Means_of_Transportation/9. Train/MVI_3169.mp4,False
496,Home,35. Photograph,Home/35. Photograph/MVI_4391.mp4,False


In [3]:
# labels = train_data['label']
# train_data_path = train_data['vedio_path']


labels = []
train_path_I50 = []

for i in range(len(train_data)):
    if train_data['include_50'][i] == True:
        labels.append(train_data['label'][i])
        train_path_I50.append("Dataset\\" + train_data['video_path'][i])
        
labels = pd.Series(labels).unique()
labels = pd.Series(labels).to_list()

train_path_I50 = pd.Series(train_path_I50)

# train_path_I50.sample(5), labels.sample(5)
labels[:5]

['1. loud', '19. House', '83. big large', '91. Priest', '23. Court']

In [4]:
label_map = dict()

for i in range(len(labels)):
    split = labels[i].split(" ")
        
    label = split[1:]
    label = " ".join(label)
    
    label_map[label] = int(split[0].split(".")[0])

labels = list(label_map.keys())
label_map    

{'loud': 1,
 'House': 19,
 'big large': 83,
 'Priest': 91,
 'Court': 23,
 'train ticket': 16,
 'it': 44,
 'Shoes': 44,
 'Dog': 1,
 'Bank': 35,
 'Thank you': 55,
 'Election': 14,
 'Cow': 5,
 'Window': 28,
 'quiet': 2,
 'dry': 97,
 'long': 78,
 'Hello': 48,
 'Bird': 4,
 'Hat': 37,
 'Black': 54,
 'short': 79,
 'White': 55,
 'Fan': 53,
 'new': 91,
 'Store or Shop': 28,
 'Monday': 67,
 'Death': 2,
 'Cell phone': 54,
 'you (plural)': 46,
 'T-Shirt': 42,
 'Girl': 78,
 'Father': 61,
 'Red': 47,
 'hot': 87,
 'Fall': 64,
 'I': 40,
 'Time': 86,
 'Car': 11,
 'Good Morning': 51,
 'Summer': 61,
 'Paint': 40,
 'Teacher': 84,
 'Brother': 66,
 'good': 94,
 'happy': 3,
 'Boy': 77,
 'small little': 84,
 'Pen': 34,
 'Year': 78}

In [5]:
# Loading all the video frames
video_frames = []

for label in tqdm(labels):
    label_videos = os.listdir("MP_data/"+label)
    window = []
    
    for video in label_videos:
        res = np.load("MP_data/" + f"{label}/" + video)
        window.append(res)
    video_frames.append(window)

print(len(video_frames))
        

  0%|          | 0/50 [00:00<?, ?it/s]

50


In [6]:
# Padding vedio sequences so all array are of equal length

import tensorflow as tf

# Checking whether the gpu is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        print("Using GPU:", gpus)

    except RuntimeError as e:
        print(e)

else:
    print("No GPU found.")




Using GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [7]:
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.utils import to_categorical

# Pad sequences to ensure they have the same length
max_length = max(len(seq) for seq in video_frames)
padded_video_frames = []

for frames in video_frames:
    padded_video_frames.append(pad_sequences(frames, maxlen=max_length, dtype='float32', padding='post', truncating='post'))

for i, frames in enumerate(padded_video_frames):
    print(f"Video {i} shape: {frames.shape}")

Video 0 shape: (13, 19, 1662)
Video 1 shape: (18, 19, 1662)
Video 2 shape: (15, 19, 1662)
Video 3 shape: (13, 19, 1662)
Video 4 shape: (15, 19, 1662)
Video 5 shape: (12, 19, 1662)
Video 6 shape: (11, 19, 1662)
Video 7 shape: (14, 19, 1662)
Video 8 shape: (12, 19, 1662)
Video 9 shape: (12, 19, 1662)
Video 10 shape: (13, 19, 1662)
Video 11 shape: (11, 19, 1662)
Video 12 shape: (16, 19, 1662)
Video 13 shape: (10, 19, 1662)
Video 14 shape: (16, 19, 1662)
Video 15 shape: (19, 19, 1662)
Video 16 shape: (14, 19, 1662)
Video 17 shape: (13, 19, 1662)
Video 18 shape: (17, 19, 1662)
Video 19 shape: (13, 19, 1662)
Video 20 shape: (14, 19, 1662)
Video 21 shape: (13, 19, 1662)
Video 22 shape: (17, 19, 1662)
Video 23 shape: (11, 19, 1662)
Video 24 shape: (16, 19, 1662)
Video 25 shape: (14, 19, 1662)
Video 26 shape: (11, 19, 1662)
Video 27 shape: (8, 19, 1662)
Video 28 shape: (9, 19, 1662)
Video 29 shape: (16, 19, 1662)
Video 30 shape: (15, 19, 1662)
Video 31 shape: (16, 19, 1662)
Video 32 shape: (16,

In [8]:
max_frames = max(video.shape[0] for video in padded_video_frames)

# Pad the number of frames for each video
X = np.zeros((len(padded_video_frames), max_frames, 19, 1662), dtype='float32')

for i, video in enumerate(padded_video_frames):
    X[i, :video.shape[0]] = video

print(X.shape)

(50, 19, 19, 1662)


In [9]:
y = np.array([label_map[label] for label in labels])
y = to_categorical(y).astype(int)

print(y.shape)

(50, 98)


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) 

print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(40, 19, 19, 1662) (10, 19, 19, 1662) (40, 98) (10, 98)


# Model

In [13]:
def downsample_frames(frames, target_height, target_width):
    """
    Downsample a batch of frames to the target resolution.

    Args:
        frames: A tensor of shape (batch_size, time_steps, height, width, channels).
        target_height: The desired height for downsampling.
        target_width: The desired width for downsampling.

    Returns:
        A tensor of downsampled frames of shape (batch_size, time_steps, target_height, target_width, channels).
    """
    batch_size, time_steps, height, width, channels = frames.shape

    # Reshape to (batch_size * time_steps, height, width, channels) for resizing each frame
    reshaped_frames = tf.reshape(frames, (-1, height, width, channels))
    
    # Resize all frames to the target resolution
    resized_frames = tf.image.resize(reshaped_frames, [target_height, target_width])
    
    # Reshape back to the original time-step structure (batch_size, time_steps, target_height, target_width, channels)
    downsampled_frames = tf.reshape(resized_frames, (batch_size, time_steps, target_height, target_width, channels))
    
    return downsampled_frames

In [15]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Flatten, TimeDistributed, Reshape
from tensorflow.keras.models import Model

tf.keras.backend.clear_session()

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

class INCLUDE50_V1(Model):
    def __init__(self, input_shape, num_classes=50, time_steps=10):
        super().__init__()
        self.num_classes = num_classes
        self.input_shape_ = input_shape
        self.time_steps = time_steps
        
        # MobileNetV2 base
        self.MobileNet = MobileNetV2(input_shape=self.input_shape_, include_top=False, weights='imagenet')
        for layer in self.MobileNet.layers[:-20]:
            layer.trainable = False  # Correctly freeze layers
        
        # Apply MobileNetV2 to each time step
        self.TimeDistribute = TimeDistributed(self.MobileNet)
        
        # Bidirectional LSTM layers
        self.BD1 = Bidirectional(LSTM(32, return_sequences=True))
        self.BD2 = Bidirectional(LSTM(32, return_sequences=True))
        
        # Flatten the output
        self.flatten = Flatten()
        
        # Fully connected layer
        self.dense = Dense(128, activation='relu')
        
        # Output layer
        self.outputs = Dense(self.num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        x = self.TimeDistribute(inputs)
        
        # Debugging shape before reshaping
        print("Shape before Reshape:", x.shape)
        
        shape = tf.shape(x)
        x = Reshape((shape[1], -1))(x)  # Reshape for LSTM
        
        # Debugging shape after reshaping
        print("Shape after Reshape:", x.shape)
        
        x = self.BD1(x)
        x = self.BD2(x)
        x = self.flatten(x)
        x = self.dense(x)
        
        return self.outputs(x)



In [17]:
# Invalid no. of classes are expected while splitting the data (Y_train)
# Downsampling the imput shape
# Input shape is invalid
# Properly preprocessing input data (X_train) 
num_classes = 50
input_shape = (1920, 1080, 3)
time_steps = 19
batch_size = 32

target_height, target_width = 224, 224

input_layer = tf.tensor(batch_size , time_steps , *input_shape)

downsampled_input = downsample_frames(input_layer, target_height, target_width)

model = INCLUDE50_V1(input_shape=input_shape, num_classes=num_classes, time_steps=time_steps)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
 
model.build(input_shape=downsampled_input)


model.summary()


AttributeError: 'tuple' object has no attribute 'shape'

In [13]:
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32)

Epoch 1/10


ValueError: in user code:

    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1160, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1146, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 1135, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\engine\training.py", line 993, in train_step
        y_pred = self(x, training=True)
    File "c:\Users\swast\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "C:\Users\swast